In [ ]:
# streamlit app = app.py
#figures codes are available there

In [2]:
#importing libraries
import pandas as pd
import plotly.express as px



In [3]:
# Load CSV
df = pd.read_csv("/content/drive/MyDrive/unified project/hhs care2/HHS_Unaccompanied_Alien_Children_Program (1).csv")

In [4]:
df.tail()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
1165,NaN,NaN,NaN,NaN,NaN,NaN
1166,NaN,NaN,NaN,NaN,NaN,NaN
1167,NaN,NaN,NaN,NaN,NaN,NaN
1168,NaN,NaN,NaN,NaN,NaN,NaN
1169,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df =df.dropna(subset=['Date'])

In [6]:
# Convert date
df["Date"] = pd.to_datetime(df["Date"])

# Sort chronologically
df = df.sort_values("Date")

# Create complete daily index
full_index = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")

df = df.set_index("Date").reindex(full_index)

df.index.name = "date"

In [7]:
df.head()

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
date,,,,,
2023-01-12,33.0,53.0,34.0,"6,566",436.0
2023-01-13,NaN,NaN,NaN,NaN,NaN
2023-01-14,NaN,NaN,NaN,NaN,NaN
2023-01-15,NaN,NaN,NaN,NaN,NaN
2023-01-16,NaN,NaN,NaN,NaN,NaN


In [8]:
# Fill missing numeric values if needed
df = df.fillna(method="ffill")

/tmp/ipykernel_1407/1743183121.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")


In [9]:
df.head()

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
date,,,,,
2023-01-12,33.0,53.0,34.0,"6,566",436.0
2023-01-13,33.0,53.0,34.0,"6,566",436.0
2023-01-14,33.0,53.0,34.0,"6,566",436.0
2023-01-15,33.0,53.0,34.0,"6,566",436.0
2023-01-16,33.0,53.0,34.0,"6,566",436.0


In [10]:
#2. Data Quality & Validation
#Detect Missing / Duplicate Dates
# Missing dates
missing_dates = full_index.difference(df.index)

# Duplicate dates
duplicates = df.index[df.index.duplicated()]

In [11]:
#Removing comma, converting numeric column to float
df['Children in HHS Care'] = df['Children in HHS Care'].str.replace(',', '').astype(float)

In [12]:
#Logical Constraints
# Transfers should not exceed CBP custody
df["flag_transfer_error"] = df["Children transferred out of CBP custody"] > df["Children in CBP custody"]

# Discharges should not exceed HHS care
df["flag_discharge_error"] = df["Children discharged from HHS Care"] >  df["Children in HHS Care"]

In [13]:
#Reporting Anomalies
df["anomaly_flag"] = df[
    ["flag_transfer_error", "flag_discharge_error"]
].any(axis=1)

In [14]:
df.head(16)

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,flag_transfer_error,flag_discharge_error,anomaly_flag
date,,,,,,,,
2023-01-12,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-13,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-14,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-15,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-16,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-17,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-18,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-19,33.0,53.0,34.0,6566.0,436.0,False,False,False
2023-01-20,33.0,53.0,34.0,6566.0,436.0,False,False,False


In [15]:
#3. Derived Healthcare Capacity Metrics
#Total System Load
#Total Load = CBP Custody + HHS Care
df["total_system_load"] = df["Children in CBP custody"] + df["Children in HHS Care"]

In [16]:
#Net Daily Intake
#Net Intake = Transfers to HHS − Discharges
df["net_intake"] = df["Children transferred out of CBP custody"] - df["Children discharged from HHS Care"]

In [17]:
#Care Load Growth Rate
df["care_load_growth_rate"] = df["Children in HHS Care"].pct_change()

In [18]:
#Backlog Indicator
df["backlog_indicator"] = df["net_intake"].rolling(7).sum()

#Positive value → backlog building.

In [19]:
df.head(18)

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,flag_transfer_error,flag_discharge_error,anomaly_flag,total_system_load,net_intake,care_load_growth_rate,backlog_indicator
date,,,,,,,,,,,,
2023-01-12,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,NaN,NaN
2023-01-13,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN
2023-01-14,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN
2023-01-15,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN
2023-01-16,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN
2023-01-17,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN
2023-01-18,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,-2814.0
2023-01-19,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,-2814.0
2023-01-20,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,-2814.0


In [20]:
#4. Trend & Temporal Analysis
#Weekly / Monthly Trends
weekly = df.resample("W").mean()
monthly = df.resample("M").mean()

/tmp/ipykernel_1407/3570798157.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.resample("M").mean()


In [21]:
#Early vs Late Comparison
mid_point = len(df) // 2

early_period = df.iloc[:mid_point]
late_period = df.iloc[mid_point:]

comparison = pd.DataFrame({
    "Early Avg Load": early_period["total_system_load"].mean(),
    "Late Avg Load": late_period["total_system_load"].mean()
}, index=["Comparison"])

In [22]:
#5. Pressure & Stress Identification
#Rolling Averages
df["load_7d_avg"] = df["total_system_load"].rolling(7).mean()
df["load_14d_avg"] = df["total_system_load"].rolling(14).mean()

In [23]:
#Variability Analysis
df["volatility"] = df["total_system_load"].rolling(7).std()

In [24]:
#Strain Windows
strain_threshold = df["total_system_load"].mean() + df["total_system_load"].std()

df["high_strain"] = df["total_system_load"] > strain_threshold

In [25]:
#6. KPI Calculations
#Total Children Under Care
kpi_total_children = df["total_system_load"].iloc[-1]
#Net Intake Pressure
kpi_net_intake_pressure = df["net_intake"].mean()
#Care Load Volatility Index
kpi_volatility = df["total_system_load"].std()
#Backlog Accumulation Rate
kpi_backlog = df["net_intake"].rolling(14).sum().mean()
#Discharge Offset Ratio
#Discharge Offset Ratio = Discharges / Transfers
df["discharge_offset_ratio"] = df["Children discharged from HHS Care"] / df["Children transferred out of CBP custody"]

kpi_offset_ratio = df["discharge_offset_ratio"].mean()



In [26]:
df

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,flag_transfer_error,flag_discharge_error,anomaly_flag,total_system_load,net_intake,care_load_growth_rate,backlog_indicator,load_7d_avg,load_14d_avg,volatility,high_strain,discharge_offset_ratio
date,,,,,,,,,,,,,,,,,
2023-01-12,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,NaN,NaN,NaN,NaN,NaN,False,12.823529
2023-01-13,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
2023-01-14,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
2023-01-15,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
2023-01-16,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-17,7.0,31.0,11.0,2481.0,10.0,False,False,False,2512.0,1.0,0.005267,2.0,2499.285714,2482.285714,16.049032,False,0.909091
2025-12-18,11.0,50.0,6.0,2472.0,16.0,False,False,False,2522.0,-10.0,-0.003628,-7.0,2504.714286,2486.928571,16.438920,False,2.666667
2025-12-19,11.0,50.0,6.0,2472.0,16.0,False,False,False,2522.0,-10.0,0.000000,-16.0,2510.142857,2491.571429,14.633621,False,2.666667


In [27]:
df.index = pd.to_datetime(df.index)
df = df.reset_index().rename(columns={"index": "date"})

In [28]:
filtered_df = df.copy()

In [29]:
filtered_df

,date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,flag_transfer_error,flag_discharge_error,anomaly_flag,total_system_load,net_intake,care_load_growth_rate,backlog_indicator,load_7d_avg,load_14d_avg,volatility,high_strain,discharge_offset_ratio
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,NaN,NaN,NaN,NaN,NaN,False,12.823529
1,2023-01-13,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
2,2023-01-14,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
3,2023-01-15,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
4,2023-01-16,33.0,53.0,34.0,6566.0,436.0,False,False,False,6619.0,-402.0,0.000000,NaN,NaN,NaN,NaN,False,12.823529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1070,2025-12-17,7.0,31.0,11.0,2481.0,10.0,False,False,False,2512.0,1.0,0.005267,2.0,2499.285714,2482.285714,16.049032,False,0.909091
1071,2025-12-18,11.0,50.0,6.0,2472.0,16.0,False,False,False,2522.0,-10.0,-0.003628,-7.0,2504.714286,2486.928571,16.438920,False,2.666667
1072,2025-12-19,11.0,50.0,6.0,2472.0,16.0,False,False,False,2522.0,-10.0,0.000000,-16.0,2510.142857,2491.571429,14.633621,False,2.666667
1073,2025-12-20,11.0,50.0,6.0,2472.0,16.0,False,False,False,2522.0,-10.0,0.000000,-25.0,2515.571429,2496.214286,9.449112,False,2.666667


In [ ]:
from google.colab import files
df.to_csv('df.csv', index=False)
files.download('df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
#plots
fig1 = px.line(
    filtered_df,
    x="date",
    y="total_system_load",
    title="Total System Load Trend"
)
fig1.show()

fig2 = px.area(
    filtered_df,
    x="date",
    y=["Children in CBP custody", "Children in HHS Care"],
    title="CBP vs HHS Load (Stacked)"
)
fig2.show()

The 'Total System Load Trend' graph shows the combined number of children in CBP (Customs and Border Protection) custody and HHS (Health and Human Services) Care over time. Looking at the trend:

Overall Decrease: The graph indicates a significant decreasing trend in the total system load from the beginning of the dataset (January 2023) to the end (December 2025).
Quantifiable Change: The 'Early Avg Load' was approximately 8624.61, while the 'Late Avg Load' was approximately 3881.33. This highlights a substantial reduction in the average total system load over the observed period.
This trend suggests that, over time, fewer unaccompanied children are being held across both CBP custody and HHS care within the observed period.

The 'CBP vs HHS Load' graphs (both the stacked area chart and the line comparison) illustrate the distribution and trends of unaccompanied children between two primary care systems: U.S. Customs and Border Protection (CBP) custody and Health and Human Services (HHS) Care.

CBP Custody: This typically represents the initial stage where children are held after apprehension at the border.
HHS Care: This is the longer-term care system, where children are placed in shelters or foster care while awaiting reunification with family or sponsors.
The graphs show how the numbers in each of these systems fluctuate over time. For example, you might observe:

Trends in each category: Whether the number of children in CBP custody or HHS care is increasing, decreasing, or remaining stable.
Relationship between the two: How transfers from CBP to HHS affect the numbers in each system. A rise in CBP numbers followed by a rise in HHS numbers could indicate efficient transfers, while a prolonged high number in CBP might suggest bottlenecks.
Overall capacity management: Together, these graphs provide insight into the capacity and flow of children through the system, highlighting periods of high intake or efficient processing.

In [33]:
fig3 = px.line(
    filtered_df,
    x="date",
    y=["Children in CBP custody", "Children in HHS Care"],
    title="CBP vs HHS Line Comparison"
)
fig3.show()

The 'CBP vs HHS Line Comparison' graph is one of the "CBP vs HHS Load" graphs. It illustrates the distribution and trends of unaccompanied children between U.S. Customs and Border Protection (CBP) custody and Health and Human Services (HHS) Care. You can observe the trends in each category over time, how transfers from CBP to HHS affect the numbers, and gain insight into the overall capacity and flow of children through the system. This directly aligns with the explanation I provided in the previous turn for the 'CBP vs HHS Load' graphs.

In [35]:
fig4 = px.bar(
    filtered_df,
    x="date",
    y="net_intake",
    title="Daily Net Intake"
)
fig4.show()

fig5 = px.line(
    filtered_df,
    x="date",
    y="backlog_indicator",
    title="Backlog Trend (Rolling)"
)
fig5.show()

The 'Daily Net Intake' graph shows the difference between 'Children transferred out of CBP custody' and 'Children discharged from HHS Care' each day.

Positive values indicate days where more children were transferred into HHS care than were discharged, suggesting an increase in the number of children requiring care within the HHS system.
Negative values indicate days where more children were discharged from HHS care than were transferred in, suggesting a decrease in the number of children in HHS care.
This graph helps in understanding the daily fluctuations in the demand for HHS care and can highlight periods of high or low intake pressure on the system.

The 'Backlog Trend (Rolling)' graph illustrates the cumulative net intake over a 7-day rolling window, which serves as a 'backlog indicator'.

*   **Positive values** on this graph indicate that, over the past seven days, more children have been transferred into the care system than have been discharged. This suggests a **building backlog** or increasing pressure on resources.
*   **Negative values** suggest that more children have been discharged than transferred in over the past seven days, indicating a **reduction in the backlog** or a decrease in system pressure.

This trend helps in identifying periods where the system is under increasing strain due to a growing number of children requiring care versus those leaving care, providing a longer-term perspective than the daily net intake.

In [41]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Bar(x=['Early Period', 'Late Period'], y=[comparison['Early Avg Load'].iloc[0], comparison['Late Avg Load'].iloc[0]])])
fig.update_layout(title='Comparison of Average Total System Load: Early vs. Late Periods',
                  xaxis_title='Period',
                  yaxis_title='Average Total System Load')
fig.show()